<a href="https://colab.research.google.com/github/DianaDoosti-PouyanBahmani/Intelligent-Systems/blob/main/app_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from math import gcd
from functools import reduce

def lcm(a, b):
    """محاسبه کمترین مضرب مشترک دو عدد"""
    return abs(a*b) // gcd(a, b)

def lcm_multiple(numbers):
    """محاسبه کمترین مضرب مشترک چندین عدد"""
    return reduce(lcm, numbers)

def calculate_effective_interest_rate(nominal_rate, compounding_periods):
    """محاسبه نرخ بهره مؤثر سالیانه"""
    return ((1 + nominal_rate/compounding_periods) ** compounding_periods) - 1

def calculate_p_from_a(A, i, n):
    """تبدیل A به P"""
    if i == 0:
        return A * n
    return A * (((1+i)**n - 1) / (i * (1+i)**n))

def calculate_p_from_f(F, i, n):
    """تبدیل F به P"""
    if i == 0:
        return F
    return F / ((1+i)**n)

def calculate_npw_finite_equal_life(projects, i):
    """محاسبه NPW برای عمر محدود یکسان"""
    results = {}
    calculations = {}

    for proj_name, proj_data in projects.items():
        n = proj_data['life']
        total_p = 0
        calc_details = []

        # هزینه‌های فعلی
        for p_value in proj_data['present_values']:
            total_p += p_value
            calc_details.append(f"P = {p_value:,.0f}")

        # جریان‌های سالیانه ثابت
        for a_value in proj_data['annual_values']:
            p_converted = calculate_p_from_a(a_value, i, n)
            total_p += p_converted
            calc_details.append(f"A = {a_value:,.0f} → P = {p_converted:,.0f}")

        # ارزش اسقاطی
        for f_value in proj_data['salvage_values']:
            p_converted = calculate_p_from_f(f_value, i, n)
            total_p += p_converted
            calc_details.append(f"F = {f_value:,.0f} → P = {p_converted:,.0f}")

        results[proj_name] = total_p
        calculations[proj_name] = calc_details

    return results, calculations

def calculate_npw_finite_unequal_life(projects, i):
    """محاسبه NPW برای عمر محدود غیریکسان"""
    lives = [proj_data['life'] for proj_data in projects.values()]
    lcm_period = lcm_multiple(lives)

    results = {}
    calculations = {}

    for proj_name, proj_data in projects.items():
        n = proj_data['life']
        repetitions = lcm_period // n
        total_p = 0
        calc_details = []

        # محاسبه NPW برای یک دوره
        single_period_p = 0

        # هزینه‌های فعلی (فقط در شروع هر دوره)
        initial_cost = sum(proj_data['present_values'])

        # جریان‌های سالیانه ثابت
        for a_value in proj_data['annual_values']:
            p_converted = calculate_p_from_a(a_value, i, n)
            single_period_p += p_converted

        # ارزش اسقاطی در انتهای هر دوره
        salvage_value = sum(proj_data['salvage_values'])

        # محاسبه NPW کل برای LCM دوره
        for rep in range(repetitions):
            start_year = rep * n
            # هزینه اولیه در شروع هر دوره
            total_p += initial_cost * ((1+i)**(-start_year))
            # درآمدها و هزینه‌های عملیاتی
            total_p += single_period_p * ((1+i)**(-start_year))
            # ارزش اسقاطی در انتهای هر دوره
            end_year = start_year + n
            total_p += salvage_value * ((1+i)**(-end_year))

        calc_details.append(f"دوره LCM: {lcm_period} سال")
        calc_details.append(f"تعداد تکرار: {repetitions}")
        calc_details.append(f"NPW یک دوره: {single_period_p + initial_cost + salvage_value * ((1+i)**(-n)):,.0f}")

        results[proj_name] = total_p
        calculations[proj_name] = calc_details

    return results, calculations

def calculate_npw_infinite_life(projects, i):
    """محاسبه NPW برای عمر نامحدود"""
    results = {}
    calculations = {}

    for proj_name, proj_data in projects.items():
        total_p = 0
        calc_details = []

        # هزینه‌های فعلی
        for p_value in proj_data['present_values']:
            total_p += p_value
            calc_details.append(f"P = {p_value:,.0f}")

        # جریان‌های سالیانه ثابت (P = A/i)
        for a_value in proj_data['annual_values']:
            if i == 0:
                calc_details.append("خطا: نرخ بهره نمی‌تواند صفر باشد برای عمر نامحدود")
                continue
            p_converted = a_value / i
            total_p += p_converted
            calc_details.append(f"A = {a_value:,.0f} → P = {p_converted:,.0f} (A/i)")

        # ارزش اسقاطی در عمر نامحدود مجاز نیست
        if proj_data['salvage_values']:
            calc_details.append("هشدار: ارزش اسقاطی در عمر نامحدود نادیده گرفته شد")

        results[proj_name] = total_p
        calculations[proj_name] = calc_details

    return results, calculations

# تنظیمات صفحه
st.set_page_config(
    page_title="تحلیل سرمایه‌گذاری - روش NPV",
    page_icon="📈",
    layout="wide"
)

# عنوان اصلی
st.title("📊 تحلیل سرمایه‌گذاری و ارزیابی پروژه - روش NPV")
st.markdown("---")

# بخش تنظیمات کلی
st.header("🔧 تنظیمات پایه")

col1, col2, col3 = st.columns(3)

with col1:
    nominal_rate = st.number_input(
        "نرخ بازده مورد انتظار (%)",
        min_value=0.0,
        max_value=100.0,
        value=12.0,
        step=0.5
    ) / 100

with col2:
    compounding_period = st.selectbox(
        "دوره محاسبه بهره",
        options=[1, 2, 4, 12],
        format_func=lambda x: {"1": "سالیانه", "2": "شش‌ماهه", "4": "فصلی", "12": "ماهیانه"}[str(x)],
        index=0
    )

with col3:
    life_type = st.radio(
        "نوع دوره پروژه",
        options=["محدود", "نامحدود"],
        horizontal=True
    )

# محاسبه نرخ بهره مؤثر
effective_rate = calculate_effective_interest_rate(nominal_rate, compounding_period)
st.success(f"💎 نرخ بازده مؤثر سالیانه: {effective_rate:.3%}")

st.markdown("---")

# بخش ورود اطلاعات پروژه‌ها
st.header("💼 مشخصات پروژه‌ها")

# تعداد پروژه‌ها
num_projects = st.number_input("تعداد پروژه‌های مورد بررسی", min_value=2, max_value=8, value=2)

projects = {}

for i in range(num_projects):
    st.subheader(f"🏢 پروژه شماره {i+1}")

    col1, col2 = st.columns(2)

    with col1:
        proj_name = st.text_input(f"عنوان پروژه {i+1}", value=f"گزینه {chr(65+i)}")

    with col2:
        if life_type == "محدود":
            proj_life = st.number_input(f"مدت پروژه {i+1} (سال)", min_value=1, value=8)
        else:
            proj_life = float('inf')
            st.write("⏰ مدت: نامحدود")

    # مؤلفه‌های مالی
    st.write("**📋 اجزای مالی:**")

    # هزینه‌های اولیه
    with st.expander("💰 سرمایه‌گذاری اولیه"):
        present_values = []
        initial_cost = st.number_input(f"سرمایه‌گذاری اولیه - پروژه {i+1}", value=0, step=1000)
        if initial_cost != 0:
            present_values.append(-abs(initial_cost))

    # جریان‌های نقدی سالیانه
    with st.expander("💵 جریان‌های نقدی سالیانه"):
        annual_values = []
        annual_revenue = st.number_input(f"درآمد خالص سالیانه - پروژه {i+1}", value=0, step=1000)
        if annual_revenue != 0:
            annual_values.append(annual_revenue)

        operating_cost = st.number_input(f"هزینه‌های عملیاتی سالیانه - پروژه {i+1}", value=0, step=1000)
        if operating_cost != 0:
            annual_values.append(-abs(operating_cost))

        maintenance_cost = st.number_input(f"هزینه‌های نگهداری سالیانه - پروژه {i+1}", value=0, step=1000)
        if maintenance_cost != 0:
            annual_values.append(-abs(maintenance_cost))

    # ارزش اسقاطی
    with st.expander("🏦 ارزش اسقاطی"):
        salvage_values = []
        if life_type == "محدود":
            salvage_value = st.number_input(f"ارزش اسقاطی - پروژه {i+1}", value=0, step=1000)
            if salvage_value != 0:
                salvage_values.append(salvage_value)
        else:
            st.write("⚠️ در پروژه‌های نامحدود، ارزش اسقاطی لحاظ نمی‌شود.")

    projects[proj_name] = {
        "life": proj_life,
        "present_values": present_values,
        "annual_values": annual_values,
        "salvage_values": salvage_values
    }

    st.markdown("---")

# بخش محاسبات و نتایج
if st.button("🚀 محاسبه و تحلیل NPV", type="primary"):
    if len(projects) < 2:
        st.error("⚠️ حداقل دو پروژه برای مقایسه لازم است!")
    else:
        st.header("📈 نتایج تحلیل")

        # تشخیص نوع تحلیل
        if life_type == "نامحدود":
            npw_results, calculations = calculate_npw_infinite_life(projects, effective_rate)
            analysis_type = "تحلیل پروژه‌های نامحدود"
        else:
            lives = [proj_data['life'] for proj_data in projects.values()]
            if len(set(lives)) == 1:
                npw_results, calculations = calculate_npw_finite_equal_life(projects, effective_rate)
                analysis_type = "تحلیل پروژه‌های هم‌مدت"
            else:
                npw_results, calculations = calculate_npw_finite_unequal_life(projects, effective_rate)
                analysis_type = "تحلیل پروژه‌های مختلف‌المدت"

        st.info(f"🔍 روش تحلیل: {analysis_type}")

        # نمایش محاسبات تفصیلی
        st.subheader("🧮 جزئیات محاسبات")

        for proj_name, calc_details in calculations.items():
            with st.expander(f"محاسبات {proj_name}"):
                for detail in calc_details:
                    st.write(f"▫️ {detail}")
                st.write(f"**🎯 NPV نهایی: {npw_results[proj_name]:,.0f} تومان**")

        # جدول مقایسه
        st.subheader("📊 جدول مقایسه نتایج")

        df_results = pd.DataFrame([
            {"نام پروژه": proj_name, "NPV (تومان)": f"{npw:,.0f}", "وضعیت": "✅ مطلوب" if npw > 0 else "❌ نامطلوب"}
            for proj_name, npw in sorted(npw_results.items(), key=lambda x: x[1], reverse=True)
        ])

        st.dataframe(df_results, use_container_width=True)

2025-12-27 15:35:34.050 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-27 15:35:34.053 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-27 15:35:34.310 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2025-12-27 15:35:34.312 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-27 15:35:34.313 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-27 15:35:34.316 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-27 15:35:34.317 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn